# AFIM Block 4 — Agentic Financial Intelligence Monitor
**Run cells in order. Read each markdown header before running.**

## CELL 1 — Install packages + all imports

In [ ]:
# ── CELL 1: pip install + imports ──────────────────────────────────────────
!pip install yfinance pandas-ta pydantic google-generativeai --quiet

import os
import json
import time
import datetime
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta as ta

from pydantic import BaseModel
import google.generativeai as genai

print('✅ All packages installed and imported.')

## CELL 2 — Mount Drive + path constants + folder setup

In [ ]:
# ── CELL 2: Drive mount + constants ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

BASE      = '/content/drive/MyDrive/AFIM/data'
PROCESSED = '/content/drive/MyDrive/AFIM/data/processed'
UPLOADS   = '/content/drive/MyDrive/AFIM/uploads'

for folder in [BASE, PROCESSED, UPLOADS]:
    os.makedirs(folder, exist_ok=True)

TICKERS = ['RELIANCE.NS', 'TCS.NS']

print('✅ Drive mounted. Folders ready.')
print(f'   BASE      → {BASE}')
print(f'   PROCESSED → {PROCESSED}')
print(f'   UPLOADS   → {UPLOADS}')

## CELL 3 — Section 0: Column rename (Fact_Technicals + Dim_Ticker fix)

In [ ]:
# ── CELL 3: Section 0 — Column rename ──────────────────────────────────────
# ── Fact_Technicals.csv ────────────────────────────────────────────────────
ft_path = os.path.join(BASE, 'Fact_Technicals.csv')

if os.path.exists(ft_path):
    ft = pd.read_csv(ft_path)
    print('Fact_Technicals before rename:', ft.columns.tolist())

    rename_map = {}
    if 'Trade_Date' in ft.columns: rename_map['Trade_Date'] = 'Date'
    if 'Ticker_ID'  in ft.columns: rename_map['Ticker_ID']  = 'Ticker'
    if 'Close'      in ft.columns: rename_map['Close']      = 'Close_Price'

    ft.rename(columns=rename_map, inplace=True)
    ft.to_csv(ft_path, index=False)
    print('Fact_Technicals after rename :', ft.columns.tolist())
else:
    print('Fact_Technicals.csv not found — will be created in Cell 5.')

# ── Dim_Ticker.csv ─────────────────────────────────────────────────────────
dt_path = os.path.join(BASE, 'Dim_Ticker.csv')

if os.path.exists(dt_path):
    dt = pd.read_csv(dt_path)
    print('\nDim_Ticker before fix:', dt.columns.tolist())

    if 'Ticker_ID' in dt.columns:
        dt.drop(columns=['Ticker_ID'], inplace=True)
    if 'Ticker_Symbol' in dt.columns:
        dt.rename(columns={'Ticker_Symbol': 'Ticker'}, inplace=True)
else:
    # Create from scratch if missing
    dt = pd.DataFrame(columns=['Ticker', 'Company_Name', 'Exchange', 'Sector'])

# Ensure both tickers exist
defaults = {
    'RELIANCE.NS': ('Reliance Industries Ltd', 'NSE', 'Energy'),
    'TCS.NS':      ('Tata Consultancy Services', 'NSE', 'Technology'),
}
for tkr, (name, exch, sect) in defaults.items():
    if tkr not in dt['Ticker'].values:
        dt = pd.concat([dt, pd.DataFrame([{
            'Ticker': tkr, 'Company_Name': name,
            'Exchange': exch, 'Sector': sect
        }])], ignore_index=True)

# Add Market_Cap_Cr if missing
market_cap = {'RELIANCE.NS': 1850000, 'TCS.NS': 1450000}
if 'Market_Cap_Cr' not in dt.columns:
    dt['Market_Cap_Cr'] = dt['Ticker'].map(market_cap)
else:
    dt['Market_Cap_Cr'] = dt['Ticker'].map(market_cap).fillna(dt['Market_Cap_Cr'])

# Final column order
dt = dt[['Ticker', 'Company_Name', 'Exchange', 'Sector', 'Market_Cap_Cr']]
dt.to_csv(dt_path, index=False)
print('Dim_Ticker after fix:', dt.columns.tolist())
print(dt)
print('\n✅ Section 0 complete.')

## CELL 4 — Pydantic schema + parse_and_validate()

In [ ]:
# ── CELL 4: Pydantic schema + parse helper ─────────────────────────────────

class FundamentalFinal(BaseModel):
    Ticker:              str
    Fiscal_Year:         str    # 'FY2023' or 'FY2024'
    Report_Date:         str    # DD-MM-YYYY
    Total_Revenue_Cr:    float
    EBITDA_Cr:           float
    Net_Income_Cr:       float
    Net_Debt_Cr:         float  # Negative = net cash
    EBITDA_Margin_Pct:   float  # e.g. 17.6 not 0.176
    Margin_Trend_Note:   str    # e.g. 'FY23:17.6% → FY24:19.3%'


def parse_and_validate(raw_text: str):
    """Strip markdown fences, parse JSON, validate against FundamentalFinal."""
    try:
        clean = raw_text.strip()
        # Strip triple backtick fences
        if clean.startswith('`'):
            clean = clean.split('`')[1]
        if clean.lower().startswith('json'):
            clean = clean[4:]
        clean = clean.strip()
        parsed    = json.loads(clean)
        validated = FundamentalFinal(**parsed)
        return validated.dict()
    except Exception as e:
        print(f'❌ Validation failed: {e}')
        print(f'   Raw response was: {raw_text[:500]}')
        return None

print('✅ FundamentalFinal schema + parse_and_validate() ready.')

## CELL 5 — Section 1: yfinance Path A (indicators + router log)

In [ ]:
# ── CELL 5: Section 1 — yfinance + indicators ──────────────────────────────

def write_router_log(route_type, input_type, input_value, is_correct,
                     start_ts, end_ts, duration_min):
    """Append one row to Dim_Router_Log.csv with auto-increment ID."""
    log_path = os.path.join(BASE, 'Dim_Router_Log.csv')
    row = {
        'Route_Type':        route_type,
        'Input_Type':        input_type,
        'Input_Value':       input_value,
        'Is_Correct':        is_correct,
        'Request_Timestamp': start_ts.strftime('%d-%m-%Y %H:%M:%S'),
        'Output_Timestamp':  end_ts.strftime('%d-%m-%Y %H:%M:%S'),
        'Duration_Min':      round(duration_min, 2),
    }
    if os.path.exists(log_path):
        log_df = pd.read_csv(log_path)
        row['Request_ID'] = int(log_df['Request_ID'].max()) + 1
    else:
        log_df = pd.DataFrame()
        row['Request_ID'] = 1
    new_row = pd.DataFrame([row])
    log_df  = pd.concat([log_df, new_row], ignore_index=True)
    # Enforce column order
    cols = ['Request_ID','Route_Type','Input_Type','Input_Value',
            'Is_Correct','Request_Timestamp','Output_Timestamp','Duration_Min']
    log_df = log_df[[c for c in cols if c in log_df.columns]]
    log_df.to_csv(log_path, index=False)


def fetch_and_enrich(ticker: str, period: str = '2y') -> pd.DataFrame:
    """Download OHLCV, flatten MultiIndex, add all indicators."""
    df = yf.download(ticker, period=period, auto_adjust=True, progress=False)

    # ── Flatten MultiIndex columns (yfinance >= 0.2 fix) ─────────────────
    df.columns = [col[0] if isinstance(col, tuple) else col
                  for col in df.columns]

    # ── Rename columns to schema names ───────────────────────────────────
    df.rename(columns={'Close': 'Close_Price'}, inplace=True)
    df.reset_index(inplace=True)
    df.rename(columns={'index': 'Date', 'Datetime': 'Date',
                       'date': 'Date'}, inplace=True)
    if 'Date' not in df.columns:
        df.rename(columns={df.columns[0]: 'Date'}, inplace=True)

    df['Ticker'] = ticker

    # ── Drop zero-volume rows (NSE artefacts) ─────────────────────────────
    df = df[df['Volume'] > 0].copy()

    # ── Indicators ───────────────────────────────────────────────────────
    df['RSI'] = ta.rsi(df['Close_Price'], length=14)

    macd_df = ta.macd(df['Close_Price'], fast=12, slow=26, signal=9)
    df['MACD']        = macd_df['MACD_12_26_9']
    df['MACD_Hist']   = macd_df['MACDh_12_26_9']
    df['Signal_Line'] = macd_df['MACDs_12_26_9']

    df['SMA_50']  = ta.sma(df['Close_Price'], length=50)
    df['SMA_200'] = ta.sma(df['Close_Price'], length=200)

    # ── Vol_Pattern (manual) ─────────────────────────────────────────────
    df['Vol_20'] = df['Volume'].rolling(20).mean()
    df['Vol_50'] = df['Volume'].rolling(50).mean()
    df['Vol_Pattern'] = df.apply(
        lambda r: 'Expanding' if r['Vol_20'] > r['Vol_50'] else 'Drying Up',
        axis=1
    )
    df.drop(columns=['Vol_20', 'Vol_50'], inplace=True)

    # ── Drop warm-up rows ─────────────────────────────────────────────────
    df.dropna(subset=['RSI', 'SMA_200'], inplace=True)

    # ── Data types ───────────────────────────────────────────────────────
    float_cols = ['Open','High','Low','Close_Price',
                  'RSI','MACD','Signal_Line','SMA_50','SMA_200']
    for c in float_cols:
        df[c] = df[c].astype(float).round(2)
    df['MACD_Hist'] = df['MACD_Hist'].astype(float).round(4)
    df['Volume']    = df['Volume'].astype(int)

    # ── Date format ───────────────────────────────────────────────────────
    df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%d-%m-%Y')

    # ── Final column order ────────────────────────────────────────────────
    final_cols = ['Date','Ticker','Open','High','Low','Close_Price',
                  'Volume','RSI','MACD','Signal_Line','MACD_Hist',
                  'SMA_50','SMA_200','Vol_Pattern']
    df = df[final_cols]
    return df


# ── Main loop ────────────────────────────────────────────────────────────────
ft_path   = os.path.join(BASE, 'Fact_Technicals.csv')
all_frames = []

for ticker in TICKERS:
    t_start = datetime.datetime.now()
    print(f'⏳ Fetching {ticker} ...')
    try:
        df_ticker = fetch_and_enrich(ticker, period='2y')
        all_frames.append(df_ticker)
        t_end = datetime.datetime.now()
        dur   = (t_end - t_start).total_seconds() / 60
        print(f'   ✅ {ticker}: {len(df_ticker)} rows fetched.')
        write_router_log('Technical','Ticker', ticker, True, t_start, t_end, dur)
    except Exception as e:
        t_end = datetime.datetime.now()
        print(f'   ❌ {ticker} failed: {e}')
        write_router_log('Technical','Ticker', ticker, False,
                         t_start, t_end, 0.0)
    time.sleep(1)

if all_frames:
    new_df = pd.concat(all_frames, ignore_index=True)

    # ── Merge with existing file (append + dedup) ─────────────────────────
    if os.path.exists(ft_path):
        existing = pd.read_csv(ft_path)
        # Ensure old file also has Close_Price not Close
        if 'Close' in existing.columns and 'Close_Price' not in existing.columns:
            existing.rename(columns={'Close': 'Close_Price'}, inplace=True)
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df

    combined.drop_duplicates(subset=['Date','Ticker'], keep='last', inplace=True)
    combined.to_csv(ft_path, index=False)

    # ── Test signal ───────────────────────────────────────────────────────
    print(f'\n📊 Fact_Technicals shape : {combined.shape}')
    print(f'   Columns              : {combined.columns.tolist()}')
    print(f'   dtypes:\n{combined.dtypes}')
    print('\nTCS.NS last 3 rows:')
    print(combined[combined['Ticker']=='TCS.NS'].tail(3).to_string())
    print('\n✅ Section 1 complete.')

## CELL 6 — Section 5: Dim_Date generation

In [ ]:
# ── CELL 6: Section 5 — Dim_Date ───────────────────────────────────────────
start_date = datetime.date(2023, 1, 1)
end_date   = datetime.date.today()

date_range = pd.date_range(start=start_date, end=end_date, freq='D')

def get_quarter(month):
    if month <= 3:  return 'Q1'
    if month <= 6:  return 'Q2'
    if month <= 9:  return 'Q3'
    return 'Q4'

dim_date = pd.DataFrame({
    'Date':        date_range.strftime('%d-%m-%Y'),
    'Year':        date_range.year.astype(int),
    'Quarter':     [get_quarter(m) for m in date_range.month],
    'Month_Num':   date_range.month.astype(int),
    'Month_Name':  date_range.strftime('%b'),
    'Day_of_Week': date_range.strftime('%a'),
    'Is_Month_End': ['Yes' if d == d + pd.offsets.MonthEnd(0) else 'No'
                     for d in date_range],
})

dim_date_path = os.path.join(BASE, 'Dim_Date.csv')
dim_date.to_csv(dim_date_path, index=False)

print(f'✅ Dim_Date generated: {dim_date.shape[0]} rows, {dim_date.shape[1]} columns')
print(f'   Date range: {dim_date["Date"].iloc[0]} → {dim_date["Date"].iloc[-1]}')
print(dim_date.head(3).to_string())

## CELL 7 — Gemini pipeline (run_gemini_pipeline + run_with_cache)

In [ ]:
# ── CELL 7: Gemini wrapper ─────────────────────────────────────────────────

PROMPT = """
You are a financial data extraction assistant.
Extract the following fields from this annual report PDF.
All monetary values must be in Indian Crores (Cr).
EBITDA_Margin_Pct must be a percentage number like 17.6, not 0.176.
Net_Debt_Cr should be negative if the company has more cash than debt.
Margin_Trend_Note should compare this year's EBITDA margin to last year,
formatted like: "FY23:17.6% → FY24:19.3%"
If a value is genuinely not found, use 0.0 for numbers and "Not found" for strings.

Return ONLY a valid JSON object. No explanation. No markdown. No backticks.
Use exactly these field names:

{
  "Ticker": "<ticker symbol like RELIANCE.NS>",
  "Fiscal_Year": "<FY2023 or FY2024>",
  "Report_Date": "<DD-MM-YYYY>",
  "Total_Revenue_Cr": <number>,
  "EBITDA_Cr": <number>,
  "Net_Income_Cr": <number>,
  "Net_Debt_Cr": <number>,
  "EBITDA_Margin_Pct": <number>,
  "Margin_Trend_Note": "<string>"
}
"""


def write_fundamentals(record: dict):
    """Append validated record to Fact_Fundamentals.csv (dedup on Fiscal_Year + Ticker)."""
    ff_path = os.path.join(BASE, 'Fact_Fundamentals.csv')
    cols = ['Ticker','Fiscal_Year','Report_Date','Total_Revenue_Cr','EBITDA_Cr',
            'Net_Income_Cr','Net_Debt_Cr','EBITDA_Margin_Pct','Margin_Trend_Note']

    new_row = pd.DataFrame([record])[cols]

    if os.path.exists(ff_path):
        existing = pd.read_csv(ff_path)
        # Drop Sentiment_Score if it crept in
        existing = existing[[c for c in existing.columns if c != 'Sentiment_Score']]
        combined = pd.concat([existing, new_row], ignore_index=True)
    else:
        combined = new_row

    combined.drop_duplicates(subset=['Fiscal_Year','Ticker'],
                              keep='last', inplace=True)
    combined = combined[cols]
    combined.to_csv(ff_path, index=False)


def run_gemini_pipeline(pdf_path: str) -> dict | None:
    """
    Upload PDF to Gemini, extract fundamentals, validate, write to CSV.
    Returns validated dict on success, None on failure.
    """
    t_start = datetime.datetime.now()
    pdf_name = os.path.basename(pdf_path)

    # Derive ticker hint from filename
    ticker_hint = 'RELIANCE.NS' if 'reliance' in pdf_name.lower() else 'TCS.NS'

    print(f'📤 Uploading {pdf_name} to Gemini ...')
    pdf_file = genai.upload_file(pdf_path)

    # ── Async polling fix (mandatory) ─────────────────────────────────────
    while pdf_file.state.name != 'ACTIVE':
        time.sleep(2)
        pdf_file = genai.get_file(pdf_file.name)
    print(f'   File state: {pdf_file.state.name}')

    model    = genai.GenerativeModel('gemini-2.5-flash')
    response = model.generate_content([pdf_file, PROMPT])
    raw_text = response.text

    result = parse_and_validate(raw_text)
    if result is None:
        t_end = datetime.datetime.now()
        dur   = (t_end - t_start).total_seconds() / 60
        write_router_log('Fundamental','PDF', pdf_name, False, t_start, t_end, dur)
        return None

    # ── Write JSON to processed/ ───────────────────────────────────────────
    fy       = result.get('Fiscal_Year', 'UNKNOWN').replace('FY', 'FY')
    tkr_safe = result.get('Ticker','').replace('.','').replace(' ','_')
    # e.g. RELIANCENS_FY2024_output.json
    json_name = f"{tkr_safe}_{fy}_output.json"
    json_path = os.path.join(PROCESSED, json_name)
    with open(json_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'   ✅ JSON saved → {json_path}')

    # ── Write to Fact_Fundamentals.csv ─────────────────────────────────────
    write_fundamentals(result)

    # ── Router log ────────────────────────────────────────────────────────
    t_end = datetime.datetime.now()
    dur   = (t_end - t_start).total_seconds() / 60
    write_router_log('Fundamental','PDF', pdf_name, True, t_start, t_end, dur)

    print(f'   ✅ Fundamentals extracted and saved for {result["Ticker"]} {result["Fiscal_Year"]}.')
    return result


def run_with_cache(pdf_path: str) -> dict | None:
    """
    Cache-aware entry point.
    Checks cache_index.csv first. On hit → returns cached result.
    On miss → calls run_gemini_pipeline() and updates cache.
    """
    cache_path = os.path.join(BASE, 'cache_index.csv')
    pdf_name   = os.path.basename(pdf_path)

    # ── Cache check ───────────────────────────────────────────────────────
    if os.path.exists(cache_path):
        cache = pd.read_csv(cache_path)
        hit   = cache[cache['File_Name'] == pdf_name]
        if not hit.empty:
            cached_json_path = hit.iloc[0]['Output_File_Path']
            print(f'✅ Cache HIT for {pdf_name}')
            if os.path.exists(cached_json_path):
                with open(cached_json_path) as f:
                    return json.load(f)
            else:
                print(f'   ⚠️  Cached JSON not found at {cached_json_path}. Re-running pipeline.')

    print(f'⚡ Cache MISS for {pdf_name}. Running Gemini pipeline ...')
    result = run_gemini_pipeline(pdf_path)

    if result is not None:
        # ── Update cache_index.csv ────────────────────────────────────────
        fy       = result.get('Fiscal_Year', 'UNKNOWN')
        tkr_safe = result.get('Ticker','').replace('.','').replace(' ','_')
        json_name = f'{tkr_safe}_{fy}_output.json'
        json_path = os.path.join(PROCESSED, json_name)

        new_cache_row = pd.DataFrame([{
            'Doc_Key':          pdf_name,
            'File_Name':        pdf_name,
            'Processed_Date':   datetime.date.today().isoformat(),
            'Output_File_Path': json_path,
        }])
        if os.path.exists(cache_path):
            cache = pd.read_csv(cache_path)
            cache = pd.concat([cache, new_cache_row], ignore_index=True)
        else:
            cache = new_cache_row
        cache.drop_duplicates(subset=['File_Name'], keep='last', inplace=True)
        cache.to_csv(cache_path, index=False)
        print(f'   Cache updated for {pdf_name}.')

    return result


print('✅ run_gemini_pipeline() and run_with_cache() defined.')

## CELL 8 — Section 4: Cache pre-population (flag-protected)

In [ ]:
# ── CELL 8: Section 4 — Cache pre-population ───────────────────────────────
CACHE_FLAG = os.path.join(BASE, 'cache_prepopulated.flag')

if os.path.exists(CACHE_FLAG):
    print('⏭️  Cache pre-population already done. Skipping.')
else:
    PRE_POP = [
        {
            'Ticker':            'RELIANCE.NS',
            'Fiscal_Year':       'FY2023',
            'Report_Date':       '21-04-2023',
            'Total_Revenue_Cr':  874895.0,
            'EBITDA_Cr':         153758.0,
            'Net_Income_Cr':     73670.0,
            'Net_Debt_Cr':       138000.0,
            'EBITDA_Margin_Pct': 17.6,
            'Margin_Trend_Note': 'FY22:16.1% → FY23:17.6%',
        },
        {
            'Ticker':            'TCS.NS',
            'Fiscal_Year':       'FY2023',
            'Report_Date':       '12-04-2023',
            'Total_Revenue_Cr':  225458.0,
            'EBITDA_Cr':         56640.0,
            'Net_Income_Cr':     42147.0,
            'Net_Debt_Cr':       -25000.0,
            'EBITDA_Margin_Pct': 25.1,
            'Margin_Trend_Note': 'FY22:24.5% → FY23:25.1%',
        },
        {
            'Ticker':            'TCS.NS',
            'Fiscal_Year':       'FY2024',
            'Report_Date':       '15-04-2024',
            'Total_Revenue_Cr':  240893.0,
            'EBITDA_Cr':         63200.0,
            'Net_Income_Cr':     46099.0,
            'Net_Debt_Cr':       -28000.0,
            'EBITDA_Margin_Pct': 26.2,
            'Margin_Trend_Note': 'FY23:25.1% → FY24:26.2%',
        },
    ]

    cache_rows   = []
    router_rows  = []
    fund_records = []

    # Fake historical timestamps (2-3 days ago)
    base_ts = datetime.datetime.now() - datetime.timedelta(days=3)
    durations = [2.3, 1.8, 3.1]   # minutes
    file_tags = [
        ('RELIANCE_FY2023.pdf', 'RELIANCE_FY2023.pdf', '2026-04-11'),
        ('TCS_FY2023.pdf',      'TCS_FY2023.pdf',      '2026-04-11'),
        ('TCS_FY2024.pdf',      'TCS_FY2024.pdf',      '2026-04-12'),
    ]

    for i, record in enumerate(PRE_POP):
        # 1. Validate against Pydantic
        validated = FundamentalFinal(**record).dict()

        # 2. Save JSON
        tkr_safe  = validated['Ticker'].replace('.','').replace(' ','_')
        fy        = validated['Fiscal_Year']
        json_name = f"{tkr_safe}_{fy}_output.json"
        json_path = os.path.join(PROCESSED, json_name)
        with open(json_path, 'w') as f:
            json.dump(validated, f, indent=2)
        print(f'   ✅ Saved JSON: {json_name}')

        # 3. Collect for Fact_Fundamentals
        fund_records.append(validated)

        # 4. Cache index row
        doc_key, fname, proc_date = file_tags[i]
        cache_rows.append({
            'Doc_Key':          doc_key,
            'File_Name':        fname,
            'Processed_Date':   proc_date,
            'Output_File_Path': json_path,
        })

        # 5. Router log row
        t_req = base_ts + datetime.timedelta(hours=i*2)
        t_out = t_req + datetime.timedelta(minutes=durations[i])
        router_rows.append({
            'Route_Type':        'Fundamental',
            'Input_Type':        'PDF',
            'Input_Value':       fname,
            'Is_Correct':        True,
            'Request_Timestamp': t_req.strftime('%d-%m-%Y %H:%M:%S'),
            'Output_Timestamp':  t_out.strftime('%d-%m-%Y %H:%M:%S'),
            'Duration_Min':      durations[i],
        })

    # ── Write Fact_Fundamentals ───────────────────────────────────────────
    ff_path = os.path.join(BASE, 'Fact_Fundamentals.csv')
    cols    = ['Ticker','Fiscal_Year','Report_Date','Total_Revenue_Cr','EBITDA_Cr',
               'Net_Income_Cr','Net_Debt_Cr','EBITDA_Margin_Pct','Margin_Trend_Note']
    new_ff  = pd.DataFrame(fund_records)[cols]
    if os.path.exists(ff_path):
        existing_ff = pd.read_csv(ff_path)
        existing_ff = existing_ff[[c for c in existing_ff.columns
                                    if c != 'Sentiment_Score']]
        combined_ff = pd.concat([existing_ff, new_ff], ignore_index=True)
    else:
        combined_ff = new_ff
    combined_ff.drop_duplicates(subset=['Fiscal_Year','Ticker'],
                                keep='last', inplace=True)
    combined_ff = combined_ff[cols]
    combined_ff.to_csv(ff_path, index=False)
    print(f'\n✅ Fact_Fundamentals: {combined_ff.shape[0]} rows written.')

    # ── Write cache_index ─────────────────────────────────────────────────
    cache_path = os.path.join(BASE, 'cache_index.csv')
    new_cache  = pd.DataFrame(cache_rows)
    if os.path.exists(cache_path):
        old_cache = pd.read_csv(cache_path)
        new_cache = pd.concat([old_cache, new_cache], ignore_index=True)
    new_cache.drop_duplicates(subset=['File_Name'], keep='last', inplace=True)
    new_cache.to_csv(cache_path, index=False)
    print(f'✅ cache_index: {new_cache.shape[0]} rows.')

    # ── Write router log entries ──────────────────────────────────────────
    log_path = os.path.join(BASE, 'Dim_Router_Log.csv')
    new_log  = pd.DataFrame(router_rows)
    if os.path.exists(log_path):
        old_log = pd.read_csv(log_path)
        start_id = int(old_log['Request_ID'].max()) + 1
        new_log  = pd.concat([old_log, new_log], ignore_index=True)
    else:
        start_id = 1
        new_log  = new_log
    # Re-assign IDs for the new rows
    new_log.reset_index(drop=True, inplace=True)
    new_log['Request_ID'] = range(1, len(new_log) + 1)
    log_cols = ['Request_ID','Route_Type','Input_Type','Input_Value',
                'Is_Correct','Request_Timestamp','Output_Timestamp','Duration_Min']
    new_log = new_log[[c for c in log_cols if c in new_log.columns]]
    new_log.to_csv(log_path, index=False)
    print(f'✅ Dim_Router_Log: {new_log.shape[0]} rows.')

    # ── Write flag ────────────────────────────────────────────────────────
    with open(CACHE_FLAG, 'w') as f:
        f.write(datetime.datetime.now().isoformat())
    print('\n✅ Section 4 complete. Flag written.')

## CELL 9 — LIVE DEMO: Gemini extracts RELIANCE FY2024
**Before running:** upload `reliance_fy2024.pdf` to `/AFIM/uploads/` in Drive.

In [ ]:
# ── CELL 9: LIVE DEMO CELL — Gemini wrapper ────────────────────────────────
# Set pdf_path to RELIANCE FY2024 PDF before running
# ─────────────────────────────────────────────────────────────────────────────
result = run_with_cache(
    pdf_path='/content/drive/MyDrive/AFIM/uploads/reliance_fy2024.pdf'
)

if result:
    print('\n📋 Extracted result:')
    for k, v in result.items():
        print(f'   {k}: {v}')
else:
    print('❌ Pipeline returned None — check validation error above.')

## CELL 10 — Verification: shapes, columns, contents

In [ ]:
# ── CELL 10: Verification ──────────────────────────────────────────────────
files = {
    'Fact_Technicals.csv':  os.path.join(BASE, 'Fact_Technicals.csv'),
    'Fact_Fundamentals.csv':os.path.join(BASE, 'Fact_Fundamentals.csv'),
    'Dim_Ticker.csv':       os.path.join(BASE, 'Dim_Ticker.csv'),
    'Dim_Date.csv':         os.path.join(BASE, 'Dim_Date.csv'),
    'Dim_Router_Log.csv':   os.path.join(BASE, 'Dim_Router_Log.csv'),
    'cache_index.csv':      os.path.join(BASE, 'cache_index.csv'),
}

print('=' * 70)
for name, path in files.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'\n📄 {name}')
        print(f'   Shape  : {df.shape}')
        print(f'   Columns: {df.columns.tolist()}')
        if df.shape[0] <= 10:
            print(df.to_string(index=False))
        else:
            print(df.head(3).to_string(index=False))
            print('   ... (truncated)')
    else:
        print(f'\n❌ {name} — NOT FOUND')

print('\n' + '=' * 70)
print('\n🔎 Test signal checks:')

# ── Column name checks ────────────────────────────────────────────────────
ft  = pd.read_csv(files['Fact_Technicals.csv'])
ff  = pd.read_csv(files['Fact_Fundamentals.csv'])
dt  = pd.read_csv(files['Dim_Ticker.csv'])

checks = [
    ('Fact_Technicals has Date column',        'Date'            in ft.columns),
    ('Fact_Technicals has Ticker column',      'Ticker'          in ft.columns),
    ('Fact_Technicals has Close_Price column', 'Close_Price'     in ft.columns),
    ('No Trade_Date in Fact_Technicals',       'Trade_Date'  not in ft.columns),
    ('No Ticker_ID in Fact_Technicals',        'Ticker_ID'   not in ft.columns),
    ('Dim_Ticker has Ticker column',           'Ticker'          in dt.columns),
    ('Dim_Ticker has Market_Cap_Cr',           'Market_Cap_Cr'   in dt.columns),
    ('No Ticker_ID in Dim_Ticker',             'Ticker_ID'   not in dt.columns),
    ('No Sentiment_Score in Fundamentals',     'Sentiment_Score' not in ff.columns),
    ('Fact_Technicals ~1000 rows',             len(ft) >= 800),
    ('Fact_Fundamentals >= 3 rows',            len(ff) >= 3),
    ('RSI populated',                          ft['RSI'].notna().sum() > 0),
    ('SMA_200 populated',                      ft['SMA_200'].notna().sum() > 0),
]

all_pass = True
for label, passed in checks:
    icon = '✅' if passed else '❌'
    print(f'   {icon}  {label}')
    if not passed:
        all_pass = False

print()
if all_pass:
    print('🎉 All checks passed. Block 4 complete — ready for Block 5.')
else:
    print('⚠️  Some checks failed. Review output above.')

# ── Cache HIT smoke test ──────────────────────────────────────────────────
print('\n🔎 Cache HIT smoke test (TCS_FY2024.pdf):')
smoke = run_with_cache(
    pdf_path='/content/drive/MyDrive/AFIM/uploads/TCS_FY2024.pdf'
)
# This should print "Cache HIT" even though the file doesn't exist on disk
# because we pre-populated cache_index in Cell 8
if smoke:
    print(f'   Returned: {smoke["Ticker"]} {smoke["Fiscal_Year"]}')